# 🚀 RAG-Scope-RU: Запуск в Google Colab (Tesla T4)

Ноутбук настроен для работы с вашим Google Drive:
- Монтирование Google Drive и сохранение результатов в `/content/drive/MyDrive/RAG_Scope_Experiments` (не пропадут при перезапуске сессии!)
- Распаковка архива `docs.zip` напрямую из Google Drive
- Автономный запуск сервера **Qdrant** прямо в среде Colab (без Docker)
- Проведение **A/B эксперимента (pypdf vs pymupdf4llm)** на 3 весовых категориях моделей:
  - **Small (29M)**: `cointegrated/rubert-tiny2`
  - **Base (305M)**: `deepvk/USER-bge-m3`
  - **Large (568M)**: `BAAI/bge-m3` и `intfloat/multilingual-e5-large`
- Замер метрик ранжирования (Hit@K, MRR, latency) и качества генерации (**RAGAS**: Faithfulness, AnsCorr, Context Recall)
- Построение интерактивных графиков сравнения прямо в ячейках блокнота.

### 1. Проверка GPU и подключение Google Drive

In [ ]:
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

import torch
if torch.cuda.is_available():
    print(f'✅ GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB VRAM)')
else:
    print('⚠️ Внимание: GPU не подключен. В меню выберите: Среда выполнения -> Сменить тип -> T4 GPU')

### 2. Клонирование репозитория и синхронизация папки experiments

In [ ]:
%cd /content
!rm -rf /content/RAG-Scope-RU
!git clone https://github.com/TheNarhawl/RAG-Scope-RU.git
%cd /content/RAG-Scope-RU

from pathlib import Path
# Создаем персистентную папку на Google Диске для сохранения результатов
drive_exp_dir = Path('/content/drive/MyDrive/RAG_Scope_Experiments')
drive_exp_dir.mkdir(parents=True, exist_ok=True)

# Создаем симлинк, чтобы все отчеты и метрики сразу писались на Google Диск
!rm -rf /content/RAG-Scope-RU/experiments
!ln -s /content/drive/MyDrive/RAG_Scope_Experiments /content/RAG-Scope-RU/experiments
print('✅ Папка experiments привязана к Google Drive!')

### 3. Автономный запуск сервера Qdrant в Google Colab
Скачивается официальный нативный бинарник Qdrant для Linux и стартует в фоне за несколько секунд.

In [ ]:
import os, subprocess, time, requests
from pathlib import Path

if not Path('/content/qdrant').exists():
    print('Скачивание бинарника Qdrant...')
    !wget -q -O /content/qdrant.tar.gz https://github.com/qdrant/qdrant/releases/download/v1.12.6/qdrant-x86_64-unknown-linux-gnu.tar.gz
    !tar -xzf /content/qdrant.tar.gz -C /content
    !chmod +x /content/qdrant

# Запуск в фоновом режиме
subprocess.Popen(['/content/qdrant'], stdout=open('/content/qdrant.log', 'w'), stderr=subprocess.STDOUT)

# Ожидание готовности сервера
for i in range(15):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:6333/readyz', timeout=1)
        if r.status_code == 200:
            print(f'✅ Qdrant готов к работе (через {i+1} сек)!')
            break
    except Exception:
        pass

### 4. Установка зависимостей проекта

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt
print('✅ Все зависимости установлены!')

### 5. Настройка переменных окружения (.env)
Для Tesla T4 (16 ГБ VRAM) ставим размер батча `64`. При необходимости укажите API ключ для RAGAS (DeepSeek, OpenAI, OpenRouter).

In [ ]:
env_content = (
    'QDRANT_URL=http://localhost:6333\n'
    'EMBED_BATCH_SIZE=64\n'
    '# HF_TOKEN=your_huggingface_token\n'
    '# OPENAI_API_KEY=your_key_here\n'
    '# OPENAI_BASE_URL=https://api.deepseek.com/v1\n'
    '# RAGAS_LLM_MODEL=deepseek-chat\n'
)
Path('.env').write_text(env_content, encoding='utf-8')
print('✅ .env файл сформирован!')

### 6. Подготовка данных (распаковка docs.zip с Google Drive)

In [ ]:
drive_zip = '/content/drive/MyDrive/docs.zip'

if Path(drive_zip).exists():
    !python src/pipelines/prepare_data.py --zip {drive_zip}
else:
    print(f'Файл {drive_zip} не найден на Диске. Запуск без кастомного архива...')
    !python src/pipelines/prepare_data.py

### 7. Генерация матрицы конфигураций
Создает 166 YAML-конфигов, включая A/B группы нарезки (`ab_chunking`) и парсинга (`ab_parsing`).

In [ ]:
!python make_configs.py

### 8. Запуск A/B Эксперимента: pypdf vs pymupdf4llm
Сравнивает качество извлечения на 3 весовых категориях:
1. **Small**: `rubert_tiny2` (29M)
2. **Base**: `user_bge_m3` (305M)
3. **Large**: `bge_m3` (568M) и `e5_large` (560M)

Скрипт автоматически индексирует документы, считает метрики, удаляет временные коллекции из памяти и строит сводную таблицу с расчетом дельты прироста.

In [ ]:
# Stage 1: Замер качества поиска (Hit@1, Hit@5, MRR, latency)
!python src/pipelines/run_ab_experiment.py --group ab_parsing

# Опционально Stage 2 (при наличии OPENAI_API_KEY в .env):
# !python src/pipelines/run_ab_experiment.py --group ab_parsing --with-ragas --ragas-limit 20

### 9. Интерактивная визуализация результатов A/B теста

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

exp_dir = Path('experiments')
records = []

for p in exp_dir.glob('ab_*_c512_*/metrics_*.json'):
    data = json.loads(p.read_text(encoding='utf-8'))
    cfg = data.get('config', {})
    summary = data.get('summary', {})
    hit = summary.get('hit_rate', {})
    ragas = summary.get('ragas', {})
    parser = cfg.get('ingest', {}).get('parser', 'pypdf')
    model = cfg.get('ingest', {}).get('embedding_model', '').split('/')[-1]
    
    records.append({
        'model': model,
        'parser': parser,
        'Hit@1': hit.get('@1', 0),
        'Hit@5': hit.get('@5', 0),
        'MRR': summary.get('mrr', 0),
        'ms/q': summary.get('ms_per_query', 0),
        'context_recall': ragas.get('context_recall', None),
        'faithfulness': ragas.get('faithfulness', None),
    })

if records:
    df = pd.DataFrame(records)
    print('Таблица результатов:')
    display(df.sort_values(by=['model', 'parser']))
    
    # График сравнения MRR
    pivot_mrr = df.pivot(index='model', columns='parser', values='MRR')
    fig, ax = plt.subplots(figsize=(10, 5))
    pivot_mrr.plot(kind='bar', ax=ax, colormap='coolwarm', alpha=0.85)
    ax.set_title('Влияние парсера на MRR (pypdf vs pymupdf4llm)', fontsize=14, pad=12)
    ax.set_ylabel('Mean Reciprocal Rank (MRR)', fontsize=12)
    ax.set_xlabel('Модель эмбеддингов', fontsize=12)
    ax.grid(axis='y', linestyle='--', alpha=0.7)
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()
else:
    print('Результаты не найдены. Сначала запустите ячейку 8.')

### 10. Запуск полной сетки экспериментов (Run Grid)
При необходимости вы можете запустить расчет всей матрицы или отфильтровать по конкретной модели:

In [ ]:
# Пример запуска полной сетки для одной модели:
!python src/pipelines/run_grid.py --filter user_bge_m3

# Сборка итогового RESULTS.md:
!python src/pipelines/summarize_experiments.py
!cat experiments/RESULTS.md